# Classification Model Training and Evaluation

**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Target:** `diagnosis` (`B` = benign, `M` = malignant)  
**Positive class:** Malignant (`M`, encoded as 1)  
**Split:** 80% training and 20% testing, stratified, `random_state=42`.

This notebook implements the five required classifiers on the same training/test split, evaluates the six required metrics, plots confusion matrices and ROC curves, and saves the fitted estimators. All preprocessing is fitted only on training data through Scikit-learn pipelines where scaling is required.


## 1. Imports and dataset loading

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
DATA_PATH = "breast_cancer_wisconsin_diagnostic.csv"

df = pd.read_csv(DATA_PATH)
X = df.drop(columns="diagnosis")
y = df["diagnosis"].map({"B": 0, "M": 1})

print("Dataset shape:", df.shape)
print("Target counts:")
print(df["diagnosis"].value_counts())


## 2. Stratified train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("\\nTraining class counts (0=Benign, 1=Malignant):")
print(y_train.value_counts().sort_index())
print("\\nTesting class counts (0=Benign, 1=Malignant):")
print(y_test.value_counts().sort_index())


## 3. Define the five classifiers

Scaling is included in pipelines for Logistic Regression, KNN and SVM because these algorithms are sensitive to feature scales. Decision Tree and Gaussian Naive Bayes are used without standardization. The scaler is fitted only on the training partition when each pipeline is fitted.


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Gaussian Naive Bayes": GaussianNB(),
    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    ])
}


## 4. Train models and calculate evaluation metrics

In [ ]:
results = []
fitted_models = {}
predictions = {}
probability_scores = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    fitted_models[model_name] = model
    predictions[model_name] = y_pred
    probability_scores[model_name] = y_score

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_score),
        "Precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))
results_df.to_csv("model_comparison.csv")


## 5. Classification reports

The classification report gives per-class precision, recall and F1-score. In this assignment, malignant is the positive class for the six comparison metrics above.


In [ ]:
for model_name, y_pred in predictions.items():
    print(f"\\n===== {model_name} =====")
    print(classification_report(
        y_test, y_pred,
        labels=[0, 1],
        target_names=["Benign", "Malignant"],
        zero_division=0
    ))


## 6. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (model_name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    display_cm = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Benign", "Malignant"]
    )
    display_cm.plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(model_name, fontsize=9)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 7. ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for model_name, y_score in probability_scores.items():
    RocCurveDisplay.from_predictions(
        y_test, y_score, name=model_name, ax=ax
    )

ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
ax.set_title("ROC Curves – Five Classification Models")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## 8. Save fitted models

The saved files include the preprocessing pipelines where required. The Streamlit application must load these files and evaluate uploaded test data; it must not retrain models.


In [ ]:
os.makedirs("model", exist_ok=True)

model_filenames = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Gaussian Naive Bayes": "gaussian_naive_bayes.joblib",
    "Support Vector Machine": "support_vector_machine.joblib"
}

for model_name, model in fitted_models.items():
    joblib.dump(model, os.path.join("model", model_filenames[model_name]))

print("Saved model files:")
for filename in model_filenames.values():
    print(os.path.join("model", filename))


## 9. Export held-out test data

This CSV includes all 30 input features and the true target column. It is suitable for the required Streamlit test-data upload demonstration.


In [ ]:
test_data = X_test.copy()
test_data["diagnosis"] = y_test.map({0: "B", 1: "M"})
test_data.to_csv("test_data.csv", index=False)
print("Saved test_data.csv:", test_data.shape)


## 10. Interpretation guidance

- **Accuracy:** proportion of all predictions that are correct.
- **AUC:** ability to rank malignant cases above benign cases across thresholds.
- **Precision (malignant):** proportion of predicted malignant cases that are truly malignant.
- **Recall (malignant):** proportion of actual malignant cases correctly identified.
- **F1:** harmonic mean of malignant precision and recall.
- **MCC:** balanced correlation coefficient based on all four confusion-matrix outcomes.

For a medical screening context, malignant recall is particularly important because false negatives represent malignant cases classified as benign. However, this dataset is a benchmark exercise and these models are not validated clinical diagnostic tools.
